# AI/ML Complaint Auto-Routing System - Model Training & Evaluation
This notebook demonstrates the end-to-end data science process of training models to predict:
1. **Complaint Priority** (High/Medium/Low) - Classification Task
2. **Resolution ETA** (in days) - Regression Task

We use features engineered from multi-lingual complaint texts using Sentence Transformers and concatenated categorical variables.

## 1. Environment Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import sys
sys.path.append('..')

# Load datasets
complaints_df = pd.read_csv('../data/complaints.csv')
officers_df = pd.read_csv('../data/officers.csv')
print(f'Complaints: {len(complaints_df)}, Officers: {len(officers_df)}')
complaints_df.head()

## 2. Text Embeddings & Categorical Feature Preprocessing
We will clean the complaint text, extract multilingual Sentence Transformer embeddings, and one-hot encode `category` and `location` columns.

In [ ]:
from utils.preprocessing import clean_text, generate_embeddings
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

print("Cleaning text and generating Sentence Transformer embeddings...")
cleaned_texts = complaints_df['complaint_text'].apply(clean_text).tolist()
embeddings = generate_embeddings(cleaned_texts)
print(f"Embeddings shape: {embeddings.shape}")

print("One-hot encoding category and location...")
cat_features = complaints_df[['category', 'location']]
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), ['category', 'location'])
    ]
)
encoded_cats = preprocessor.fit_transform(cat_features)
print(f"Categorical features shape: {encoded_cats.shape}")

# Combine features
X = np.hstack((embeddings, encoded_cats))
y_priority = complaints_df['priority']
y_eta = complaints_df['resolution_days']
print(f"Final design matrix X shape: {X.shape}")

## 3. Priority Prediction Model (Classification)
We train and compare **Logistic Regression** and **Random Forest Classifier** models.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

X_train, X_test, y_train, y_test = train_test_split(X, y_priority, test_size=0.2, random_state=42)

# Logistic Regression
lr_clf = LogisticRegression(max_iter=1000, random_state=42)
lr_clf.fit(X_train, y_train)
y_pred_lr = lr_clf.predict(X_test)

print("=== Logistic Regression Report ===")
print(classification_report(y_test, y_pred_lr))

# Random Forest Classifier
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)
rf_clf.fit(X_train, y_train)
y_pred_rf = rf_clf.predict(X_test)

print("=== Random Forest Classifier Report ===")
print(classification_report(y_test, y_pred_rf))

### Priority Model Comparison Summary
Based on our training run, here are the evaluation metrics comparison on the test set:


In [ ]:
# Print precalculated comparison
clf_data = [
    {
        "Metric": "Accuracy",
        "Logistic Regression": 0.9326923076923077,
        "Random Forest Classifier": 0.9230769230769231
    },
    {
        "Metric": "Precision (Weighted)",
        "Logistic Regression": 0.9338600675809978,
        "Random Forest Classifier": 0.9285927251043531
    },
    {
        "Metric": "Recall (Weighted)",
        "Logistic Regression": 0.9326923076923077,
        "Random Forest Classifier": 0.9230769230769231
    },
    {
        "Metric": "F1 Score (Weighted)",
        "Logistic Regression": 0.9324744166611636,
        "Random Forest Classifier": 0.9231647029441146
    }
]
comparison_clf = pd.DataFrame(clf_data)
comparison_clf

## 4. ETA Prediction Model (Regression)
We train and compare **Linear Regression** and **Random Forest Regressor** models.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X, y_eta, test_size=0.2, random_state=42)

# Linear Regression
lin_reg = LinearRegression()
lin_reg.fit(X_train_r, y_train_r)
y_pred_lin = lin_reg.predict(X_test_r)
print("=== Linear Regression Metrics ===")
print(f"MAE: {mean_absolute_error(y_test_r, y_pred_lin):.3f} days")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_r, y_pred_lin)):.3f} days")
print(f"R² Score: {r2_score(y_test_r, y_pred_lin):.3f}")

# Random Forest Regressor
rf_reg = RandomForestRegressor(n_estimators=100, random_state=42)
rf_reg.fit(X_train_r, y_train_r)
y_pred_rf_r = rf_reg.predict(X_test_r)
print("\n=== Random Forest Regressor Metrics ===")
print(f"MAE: {mean_absolute_error(y_test_r, y_pred_rf_r):.3f} days")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_r, y_pred_rf_r)):.3f} days")
print(f"R² Score: {r2_score(y_test_r, y_pred_rf_r):.3f}")

### ETA Model Comparison Summary
Based on our training run, here are the evaluation metrics comparison on the test set:


In [ ]:
reg_data = [
    {
        "Metric": "MAE (Days)",
        "Linear Regression": 19.616308281632183,
        "Random Forest Regressor": 2.250197115384615
    },
    {
        "Metric": "RMSE (Days)",
        "Linear Regression": 28.162242927971693,
        "Random Forest Regressor": 2.8276406068385542
    },
    {
        "Metric": "R\u00b2 Score",
        "Linear Regression": -45.93545149892725,
        "Random Forest Regressor": 0.5268324654315854
    }
]
comparison_reg = pd.DataFrame(reg_data)
comparison_reg

## 5. Saving the Best Models & FAISS Index
We serialize the best models using Joblib and construct the FAISS semantic index for complaints.

In [ ]:
# Save final models (this was already performed in setup_and_train.py)
print(f"Selected Priority Classifier: {metrics['best_clf']}")
print(f"Selected ETA Regressor: {metrics['best_reg']}")

# Verify they can load
loaded_clf = joblib.load('../models/priority_model.pkl')
loaded_reg = joblib.load('../models/eta_model.pkl')
loaded_pre = joblib.load('../models/pipeline_preprocessor.pkl')
print("All model artifacts successfully loaded and ready for production inference!")